# Perception SLM — real captioning (SigLIP + connector + pretrained SmolLM2 + LoRA)

This is the **proper** small-VLM recipe for fluent captions on a free T4: frozen pretrained SigLIP, a connector trained from scratch, and a **pretrained SmolLM2** decoder with LoRA. Language fluency is inherited from the LM; the connector + whole pipeline are built from scratch.

Set **Accelerator = GPU T4**, **Internet = On**, and add `HF_TOKEN` (write) under Add-ons → Secrets.

In [ ]:
REPO_URL = "https://github.com/Tushar-CYL/tryn-model-v1.git"
import os
os.chdir('/kaggle/working')
if not os.path.exists('repo'):
    !git clone $REPO_URL repo
os.chdir('/kaggle/working/repo'); !git pull --ff-only || true
print('cwd:', os.getcwd())

In [ ]:
!pip -q install transformers datasets webdataset peft hydra-core omegaconf einops \
    huggingface_hub wandb pyarrow
!pip -q install -e .
import sys; sys.path.insert(0, '/kaggle/working/repo/src')
import torch; print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'Enable GPU!'

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
sec = UserSecretsClient(); os.environ['HF_TOKEN'] = sec.get_secret('HF_TOKEN'); login(os.environ['HF_TOKEN'])

## Build COCO caption shards (more data = better captions)

In [ ]:
!python -m data_pipeline.build --dataset coco --limit 5000 --out data/processed/coco_train --maxcount 1000

## Train: connector + LoRA on SmolLM2 (frozen SigLIP)
First run downloads SigLIP (~350MB) + SmolLM2-135M (~270MB). Bump `steps` for better captions; switch `lm.model_name` to `HuggingFaceTB/SmolLM2-360M` for a stronger decoder.

In [ ]:
from common.config import load_config
from training.caption_lm import run_caption_lm

cfg = load_config('caption_lm')
cfg.data.shards_dir = 'data/processed/coco_train'
cfg.data.batch_size = 16
cfg.optim.steps = 3000          # raise for better captions (resume-friendly)
cfg.eval.every = 500
# cfg.lm.model_name = 'HuggingFaceTB/SmolLM2-360M'   # stronger decoder
res = run_caption_lm(cfg)
res['samples']

## Deploy the trainable weights (connector + LoRA) to HF

In [ ]:
REPO_ID = 'LNTTushar/perception-slm-caption'
!python scripts/push_to_hf.py --repo-id $REPO_ID --checkpoint outputs/caption_lm/caption_lm.pt --config caption_lm

## Notes
- The checkpoint is small (connector + LoRA only); the frozen SigLIP + SmolLM2 are re-downloaded from HF when loading.
- More steps + `SmolLM2-360M` + more data (`--limit 15000+`) → noticeably better captions.
- Resume across 12h sessions by re-running; `outputs/` persists within a session.